In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.phase5_eda import run_phase5_eda

print("Project root:", PROJECT_ROOT)

Project root: D:\downloads\project\Diabetic-Retinopathy-Screening-Project


In [2]:
eda_summary = run_phase5_eda(PROJECT_ROOT)

eda_summary

D:\downloads\project\Diabetic-Retinopathy-Screening-Project\src\phase5_eda.py:39: DtypeWarning: Columns (0: official_split, 1: quality_label_source, 2: duplicate_group, 3: duplicate_type, 4: review_status, 5: dme_label_source, 6: dme_label_scheme, 7: lesion_annotation_type) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(metadata_path)


Phase 5 EDA complete.
Figures saved to: D:\downloads\project\Diabetic-Retinopathy-Screening-Project\reports\figures\phase5_eda
Summary table saved to: D:\downloads\project\Diabetic-Retinopathy-Screening-Project\reports\tables


,dataset,images,labeled_images,usable_model_images,official_quality_labels,dme_labels,lesion_masks
0,aptos,5590,3662,3662,0,0,0
1,eyepacs,35126,35126,35126,0,0,0
2,idrid,81,81,81,0,81,81
3,messidor2,1744,1744,1744,1744,1744,0


In [3]:
import pandas as pd

metadata = pd.read_csv(
    PROJECT_ROOT / "data/metadata/unified_metadata.csv"
)

class_distribution = (
    metadata.dropna(subset=["dr_grade_name"])
    .groupby(["dataset", "dr_grade_name"])
    .size()
    .reset_index(name="images")
)

dataset_sizes = (
    metadata.groupby("dataset")
    .size()
    .reset_index(name="images")
)

resolution_summary = (
    metadata.groupby("dataset")
    .agg(
        median_width=("width", "median"),
        median_height=("height", "median"),
        minimum_width=("width", "min"),
        maximum_width=("width", "max"),
    )
    .reset_index()
)

quality_summary = (
    metadata.groupby("dataset")
    .agg(
        official_quality_labels=(
            "quality_label",
            lambda values: values.notna().sum(),
        ),
        automatic_quality_flags=(
            "quality_flag_auto",
            lambda values: (values != "pass").sum(),
        ),
    )
    .reset_index()
)

lesion_summary = (
    metadata.groupby("dataset")
    .agg(
        images_with_lesion_masks=(
            "lesion_available",
            lambda values: values.fillna(False).sum(),
        )
    )
    .reset_index()
)

print("DR class distribution")
display(class_distribution)

print("Dataset sizes")
display(dataset_sizes)

print("Image resolution summary")
display(resolution_summary)

print("Quality-label availability")
display(quality_summary)

print("Lesion-mask availability")
display(lesion_summary)

C:\Users\poorv\AppData\Local\Temp\ipykernel_28556\276901958.py:3: DtypeWarning: Columns (0: official_split, 1: quality_label_source, 2: duplicate_group, 3: duplicate_type, 4: review_status, 5: dme_label_source, 6: dme_label_scheme, 7: lesion_annotation_type) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(


DR class distribution


,dataset,dr_grade_name,images
0,aptos,Mild DR,370
1,aptos,Moderate DR,999
2,aptos,No DR,1805
3,aptos,Proliferative DR,295
4,aptos,Severe DR,193
5,eyepacs,Mild DR,2443
6,eyepacs,Moderate DR,5292
7,eyepacs,No DR,25810
8,eyepacs,Proliferative DR,708
9,eyepacs,Severe DR,873


Dataset sizes


,dataset,images
0,aptos,5590
1,eyepacs,35126
2,idrid,81
3,messidor2,1744


...

,dataset,median_width,median_height,minimum_width,maximum_width
0,aptos,1050.0,1050.0,474,4288
1,eyepacs,3888.0,2592.0,400,5184
2,idrid,4288.0,2848.0,4288,4288
3,messidor2,512.0,512.0,512,512


Quality-label availability


,dataset,official_quality_labels,automatic_quality_flags
0,aptos,0,267
1,eyepacs,0,12822
2,idrid,0,0
3,messidor2,1744,0


Lesion-mask availability


,dataset,images_with_lesion_masks
0,aptos,0
1,eyepacs,0
2,idrid,81
3,messidor2,0


In [4]:
findings = pd.DataFrame(
    [
        {
            "topic": "Class imbalance",
            "finding": (
                "Grade 0 is much more common than severe grades. "
                "Use class-aware methods during model development."
            ),
        },
        {
            "topic": "Dataset size",
            "finding": (
                "EyePACS is substantially larger than the other datasets."
            ),
        },
        {
            "topic": "Image resolution",
            "finding": (
                "Original image sizes differ across datasets; "
                "the processed 224×224 images provide a consistent input."
            ),
        },
        {
            "topic": "Quality labels",
            "finding": (
                "MESSIDOR-2 has official gradability labels. "
                "Other datasets have missing official quality labels."
            ),
        },
        {
            "topic": "Lesion masks",
            "finding": (
                "IDRiD has lesion masks; the other current datasets "
                "do not provide lesion masks."
            ),
        },
    ]
)

findings.to_csv(
    PROJECT_ROOT / "reports/tables/phase5_eda_findings.csv",
    index=False,
)

findings

,topic,finding
0,Class imbalance,Grade 0 is much more common than severe grades...
1,Dataset size,EyePACS is substantially larger than the other...
2,Image resolution,Original image sizes differ across datasets; t...
3,Quality labels,MESSIDOR-2 has official gradability labels. Ot...
4,Lesion masks,IDRiD has lesion masks; the other current data...
